# Part B — Finetune Qwen2.5-0.5B-Instruct as a Hinglish EMI Agent
**Run on Colab T4 GPU.** Runtime → Change runtime type → T4 GPU

In [ ]:
# 1. Install dependencies
!pip install -q transformers==4.51.0 peft==0.10.0 datasets==2.19.0 trl==0.8.6 bitsandbytes==0.43.0 accelerate==0.29.3 sentencepiece

In [ ]:
import json, torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer
from datasets import Dataset

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 2. Load Cleaned Data from Part A
Upload `cleaned_conversations.jsonl` from Part A, or paste its contents below.

In [ ]:
# If running on Colab, upload cleaned_conversations.jsonl first
# from google.colab import files; files.upload()

CLEAN_PATH = 'cleaned_conversations.jsonl'

def load_jsonl(path):
    records = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

convs = load_jsonl(CLEAN_PATH)
print(f'Loaded {len(convs)} cleaned conversations')

## 3. Format Data into Chat Template
We build a system prompt defining the EMI agent persona, then format each conversation as a single assistant response to the last customer turn.

In [ ]:
SYSTEM_PROMPT = (
    'Aap ek polite aur professional EMI collection agent hain jo Hinglish mein baat karte hain. '
    'Aapka kaam customers ko payment ke baare mein guide karna hai — late fees, UPI payments, '
    'callbacks, aur EMI schedules ke baare mein. Hamesha polite aur helpful rahein.'
)

def conv_to_training_example(conv):
    """
    Convert a multi-turn conversation into a single (user, assistant) training pair.
    User message = all customer turns joined; assistant = last agent turn.
    This is a simplification suitable for a small dataset.
    """
    turns = conv.get('turns', [])
    if len(turns) < 2:
        return None

    # Find last agent turn and the customer turn just before it
    agent_turns   = [t for t in turns if t['role'] == 'agent']
    customer_turns = [t for t in turns if t['role'] == 'customer']

    if not agent_turns or not customer_turns:
        return None

    user_msg      = customer_turns[-1]['text']
    assistant_msg = agent_turns[-1]['text']

    return {'user': user_msg, 'assistant': assistant_msg}

raw_examples = [conv_to_training_example(c) for c in convs]
examples = [e for e in raw_examples if e is not None]
print(f'Training examples: {len(examples)}')

In [ ]:
BASE_MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def format_prompt(example):
    messages = [
        {'role': 'system',    'content': SYSTEM_PROMPT},
        {'role': 'user',      'content': example['user']},
        {'role': 'assistant', 'content': example['assistant']},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

formatted = [format_prompt(e) for e in examples]

# Sanity-check: print one formatted example
print('--- Sample formatted prompt ---')
print(formatted[0][:500])

## 4. Load Base Model with 4-bit Quantisation

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
model.config.use_cache = False
print('Base model loaded.')

## 5. Configure LoRA
- `r=16`: rank — controls capacity of the adapter
- `lora_alpha=32`: scaling factor (2× rank is standard)
- `target_modules`: query and value projections in attention layers
- `lora_dropout=0.05`: light regularisation for small datasets

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'v_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 6. Run Training

In [ ]:
dataset = Dataset.from_dict({'text': formatted})

training_args = TrainingArguments(
    output_dir='./emi_agent_checkpoints',
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=5,
    save_strategy='epoch',
    optim='paged_adamw_8bit',
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    dataset_text_field='text',
    max_seq_length=512,
    tokenizer=tokenizer,
)

print('Starting training...')
trainer.train()
print('Training complete.')

## 7. Save LoRA Adapter

In [ ]:
ADAPTER_PATH = './emi_agent_adapter'
model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)
print(f'Adapter saved to {ADAPTER_PATH}')

## 8. Inference: Before vs After Finetuning
We compare the base model vs finetuned model on 5 test prompts.

In [ ]:
from peft import PeftModel

# Load fresh base model (not quantised, for clean comparison)
base_model_inf = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=torch.float16, device_map='auto', trust_remote_code=True
)
finetuned_model = PeftModel.from_pretrained(base_model_inf, ADAPTER_PATH)
finetuned_model.eval()

TEST_PROMPTS_DEMO = [
    'Meri EMI is mahine late ho gayi hai, kya main ab bhi pay kar sakta hoon?',
    'I cannot pay right now, please give me more time.',
    'UPI se payment karna hai, link bhejo please.',
    'Mujhe penalty ke baare mein batao, kitna extra lagega?',
    'Callback schedule karo, shaam 6 baje available hoon.',
]

def generate_response(mdl, prompt):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': prompt},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(mdl.device)
    with torch.no_grad():
        outputs = mdl.generate(**inputs, max_new_tokens=150, do_sample=False)
    return tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

# Base model (before finetuning) — disable adapter temporarily
print('=== BASE MODEL (before finetuning) ===')
with finetuned_model.disable_adapter():
    for i, prompt in enumerate(TEST_PROMPTS_DEMO, 1):
        resp = generate_response(finetuned_model, prompt)
        print(f'\nPrompt {i}: {prompt}')
        print(f'Response: {resp[:200]}')

print('\n=== FINETUNED MODEL (after finetuning) ===')
for i, prompt in enumerate(TEST_PROMPTS_DEMO, 1):
    resp = generate_response(finetuned_model, prompt)
    print(f'\nPrompt {i}: {prompt}')
    print(f'Response: {resp[:200]}')

## 9. Evaluation Scorecard

In [ ]:
import re

HINGLISH_KEYWORDS = {
    'sir', 'hoon', 'hai', 'hain', 'karo', 'kar', 'aap', 'main', 'mein', 'nahi',
    'nahin', 'theek', 'kya', 'kal', 'aaj', 'hoga', 'sakta', 'sakte', 'zaroor',
    'bilkul', 'payment', 'abhi', 'please', 'toh', 'bhi', 'se', 'ko', 'ki',
}
ON_TOPIC_KEYWORDS = {
    'payment', 'emi', 'loan', 'penalty', 'due', 'amount', 'pay', 'rupees',
    'rs', 'upi', 'bank', 'account', 'extension', 'callback', 'reminder',
    'transaction', 'fee', 'partial', 'installment', 'credit',
}

ALL_TEST_PROMPTS = [
    'Meri EMI is mahine bahut late ho gayi hai, kya main ab bhi pay kar sakta hoon?',
    'I cant pay right now, please give me some more time.',
    'Kal tak payment kar dunga, pakka promise hai.',
    'Mujhe penalty ke baare mein batao, kitna extra lagega?',
    'UPI se payment karna hai, link bhejo please.',
    'Mera loan account number kya hai?',
    'Aaj salary nahi aayi, 3 din mein karunga.',
    'Can I split my EMI into two payments this month?',
    'Callback schedule karo, shaam 6 baje available hoon.',
    'Payment already kar diya, confirmation nahi aayi.',
]

def score(response):
    words = set(re.findall(r'[a-zA-Z]+', response.lower()))
    wc    = len(response.split())
    lang  = bool(words & HINGLISH_KEYWORDS)
    topic = bool(words & ON_TOPIC_KEYWORDS)
    length = 1 <= wc <= 300
    return lang, topic, length

print('=' * 60)
print('  EMI AGENT EVALUATION SCORECARD')
print('=' * 60)
passes = 0
for i, prompt in enumerate(ALL_TEST_PROMPTS, 1):
    resp = generate_response(finetuned_model, prompt)
    lang, topic, length = score(resp)
    overall = lang and topic and length
    passes += overall
    status = 'PASS' if overall else 'FAIL'
    print(f'\nQ{i:02d}: {prompt[:60]}')
    print(f'     Language:{"✓" if lang else "✗"} | Topic:{"✓" if topic else "✗"} | Length:{"✓" if length else "✗"} => {status}')
print(f'\nFINAL: {passes}/10 PASSED')
print('=' * 60)